In [41]:
from dataset_gbm import CorrelatedGBMGenerativeDataset
import numpy as np

In [42]:
corr = np.array([
    [1.0, 0.8, 0.4],
    [0.8, 1.0, 0.2],
    [0.4, 0.2, 1.0],
])

dataset = CorrelatedGBMGenerativeDataset(
    n_paths=100000,
    n_steps=200,
    n_ts_features=3,
    s_price=100.0,
    mu=[0.05, 0.03, 0.07],
    sigma_gbm=[0.2, 0.15, 0.25],
    corr_matrix=corr,
    # return_log_returns=True
)

In [43]:
def estimate_gbm_params_multi(S, dt):
    # Log returns: shape (n_paths, n_assets, N)
    log_returns = np.diff(np.log(S), axis=-1)
    N = log_returns.shape[-1]

    # Mean and std of log returns (over time axis)
    mean_log_ret = log_returns.mean(axis=-1)  # shape (n_paths, n_assets)
    var_log_ret = log_returns.var(axis=-1, ddof=0)

    sigma_hat = np.sqrt(var_log_ret / dt)  # shape (n_paths, n_assets)
    mu_hat = (mean_log_ret / dt) + 0.5 * sigma_hat**2  # shape (n_paths, n_assets)

    # Average across all paths
    return mu_hat.mean(axis=0), sigma_hat.mean(axis=0)


In [44]:
def estimate_correlation_matrix_transposed(S):
    log_returns = np.log(S[:, :, 1:] / S[:, :, :-1])  # shape (n_paths, n_assets, N)

    # Reshape to (n_paths * N, n_assets)
    flat_returns = log_returns.transpose(0, 2, 1).reshape(-1, S.shape[1])

    # Estimate correlation
    corr_matrix = np.corrcoef(flat_returns.T)
    return corr_matrix


In [45]:
def pathwise_correlation_matrices_transposed(S):
    log_returns = np.log(S[:, :, 1:] / S[:, :, :-1])  # shape (n_paths, n_assets, N)
    n_paths, n_assets, _ = log_returns.shape

    corrs = np.empty((n_paths, n_assets, n_assets))

    for i in range(n_paths):
        corrs[i] = np.corrcoef(log_returns[i])

    return corrs  # shape (n_paths, n_assets, n_assets)


In [46]:
def undo_log_returns(log_returns, S0):
    """
    Reconstructs price paths from log returns and initial prices.

    Parameters:
        log_returns: np.ndarray of shape (n_paths, n_assets, N)
        S0: np.ndarray of shape (n_paths, n_assets) or (n_assets,)

    Returns:
        prices: np.ndarray of shape (n_paths, n_assets, N+1)
    """
    # Cumulative sum of log returns → log prices relative to S0
    log_price_rel = np.cumsum(log_returns, axis=-1)

    # Insert log(S0) at the beginning
    log_S0 = np.log(S0) * np.ones((*log_returns.shape[:2], 1))
    log_prices = np.concatenate([log_S0, log_S0 + log_price_rel], axis=-1)

    # Exponentiate to get prices
    prices = np.exp(log_prices)
    return prices


In [47]:
prices = dataset.paths
prices.shape

(100000, 3, 200)

In [48]:
log_returns = np.diff(np.log(prices), axis=-1)
log_returns.shape

(100000, 3, 199)

In [49]:
recovered_prices = undo_log_returns(log_returns, dataset.s_price)
recovered_prices.shape

(100000, 3, 200)

In [50]:
np.allclose(prices, recovered_prices)

True

In [51]:
mu, sigma = estimate_gbm_params_multi(prices, dataset.dt)
mu, sigma

(array([0.04955131, 0.02972243, 0.06945562]),
 array([0.1992653 , 0.14945198, 0.24909376]))

In [53]:
corr

array([[1. , 0.8, 0.4],
       [0.8, 1. , 0.2],
       [0.4, 0.2, 1. ]])

In [52]:
corr_est = estimate_correlation_matrix_transposed(prices)
corr_est

array([[1.        , 0.80004552, 0.39982057],
       [0.80004552, 1.        , 0.19991548],
       [0.39982057, 0.19991548, 1.        ]])

In [54]:
paths_corrs = pathwise_correlation_matrices_transposed(prices)
paths_corrs.shape

(100000, 3, 3)

In [56]:
mean_corr_matrix = np.mean(paths_corrs, axis=0)
mean_corr_matrix

array([[1.        , 0.79932236, 0.39898877],
       [0.79932236, 1.        , 0.19945079],
       [0.39898877, 0.19945079, 1.        ]])